In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs (cached in variables for reuse)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')

# Timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# --- metric.csv summaries ---
metric_rows = metric.shape[0]

# Unique cmdb_id and kpi_name (limit to first 50)
metric_unique_cmdb = pd.Series(metric['cmdb_id'].dropna().unique()[:50], name='metric_unique_cmdb_id')
metric_unique_kpi = pd.Series(metric['kpi_name'].dropna().unique()[:50], name='metric_unique_kpi_name')

# Min and max timestamp converted to UTC+8 timezone (timestamps are epoch seconds)
metric_ts = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metric_min_ts = metric_ts.min()
metric_max_ts = metric_ts.max()

# --- trace.csv summaries ---
trace_rows = trace.shape[0]

trace_unique_cmdb = pd.Series(trace['cmdb_id'].dropna().unique()[:50], name='trace_unique_cmdb_id')
trace_unique_trace_name = pd.Series(trace['trace_name'].dropna().unique()[:50], name='trace_unique_trace_name')

trace_ts = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_min_ts = trace_ts.min()
trace_max_ts = trace_ts.max()

# Display compact outputs (use variable names, separated by commas)
metric_rows, metric_unique_cmdb, metric_unique_kpi, metric_min_ts, metric_max_ts, trace_rows, trace_unique_cmdb, trace_unique_trace_name, trace_min_ts, trace_max_ts

```
Out[1]:
```
```python
# Summarize previous execution results in plain English (compact)
summary = (
    f"metric.csv: {metric_rows} rows. "
    f"Unique cmdb_id count: {metric_unique_cmdb.shape[0]} (examples: {', '.join(metric_unique_cmdb.astype(str).values[:5])}). "
    f"Unique kpi_name count (showing up to 50): {metric_unique_kpi.shape[0]} (examples: {', '.join(metric_unique_kpi.astype(str).values[:5])}). "
    f"Timestamp range (UTC+8): {metric_min_ts.strftime('%Y-%m-%d %H:%M:%S %Z')} to {metric_max_ts.strftime('%Y-%m-%d %H:%M:%S %Z')}. "
    f"trace.csv: {trace_rows} rows. "
    f"Unique cmdb_id count: {trace_unique_cmdb.shape[0]} (examples: {', '.join(trace_unique_cmdb.astype(str).values[:5])}). "
    f"Unique trace_name count (showing up to 50): {trace_unique_trace_name.shape[0]} (examples: {', '.join(trace_unique_trace_name.astype(str).values[:5])}). "
    f"Timestamp range (UTC+8): {trace_min_ts.strftime('%Y-%m-%d %H:%M:%S %Z')} to {trace_max_ts.strftime('%Y-%m-%d %H:%M:%S %Z')}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(37527, 0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
43    redis_008
44    redis_009
45    redis_010
46    redis_011
47    redis_012
Name: metric_unique_cmdb_id, Length: 48, dtype: object, 0                           ACS
1                          AIOS
2                           AWS
3                  CPU_Used_Pct
4                  CPU_free_pct
                ...            
45     container_thread_running
46       container_thread_total
47    container_thread_used_pct
48                   Agent_ping
49                 CPU_idle_pct
Name: metric_unique_kpi_name, Length: 50, dtype: object, Timestamp('2020-05-22 00:00:00+0800', tz='Asia/Shanghai'), Timestamp('2020-05-22 00:30:00+0800', tz='Asia/Shanghai'), 6024, 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
Name: trace_unique_cmdb_id, dtype: object, 0         trace.from_os_022.remoteprocess.duration_mean
1          trace.from_os_022.remoteprocess.duration_p95
2            trace.from_os_022.remoteprocess.error_rate
3             trace.from_os_022.remoteprocess.row_count
4                          trace.self.csf.duration_mean
                            ...                        
45     trace.from_docker_002.remoteprocess.duration_p95
46       trace.from_docker_002.remoteprocess.error_rate
47        trace.from_docker_002.remoteprocess.row_count
48    trace.from_docker_003.remoteprocess.duration_mean
49     trace.from_docker_003.remoteprocess.duration_p95
Name: trace_unique_trace_name, Length: 50, dtype: object, Timestamp('2020-05-22 00:00:00+0800', tz='Asia/Shanghai'), Timestamp('2020-05-22 00:29:00+0800', tz='Asia/Shanghai'))```
```

In [ ]:
# In[2]:

# Reuse existing variables: metric, tz (pytz timezone for Asia/Shanghai)
import pandas as pd
import pytz

# Ensure tz exists
tz = pytz.timezone('Asia/Shanghai')

# Build candidate sets
os_ids = [f"os_{i:03d}" for i in range(1, 23)]        # os_001 .. os_022
docker_ids = [f"docker_{i:03d}" for i in range(1, 9)] # docker_001 .. docker_008
db_ids = [f"db_{i:03d}" for i in range(1, 14)]       # db_001 .. db_013
candidates = os_ids + docker_ids + db_ids

# Filter metric rows to only those candidate cmdb_ids
metric_candidate = metric[metric['cmdb_id'].isin(candidates)].copy()

# Group and aggregate
grp = metric_candidate.groupby('cmdb_id').agg(
    row_count=('cmdb_id', 'size'),
    unique_kpi_count=('kpi_name', 'nunique'),
    min_timestamp_s=('timestamp', 'min'),
    max_timestamp_s=('timestamp', 'max')
).reset_index()

# Convert min/max timestamps (epoch seconds) to UTC+8 timezone strings
grp['min_timestamp'] = pd.to_datetime(grp['min_timestamp_s'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
grp['max_timestamp'] = pd.to_datetime(grp['max_timestamp_s'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and sort, limit to top 20 by row_count descending
summary_df = grp.loc[:, ['cmdb_id', 'row_count', 'unique_kpi_count', 'min_timestamp', 'max_timestamp']].sort_values('row_count', ascending=False).head(20)

# Compute missing candidate components (from the candidate sets but not present in metric.csv)
present = metric['cmdb_id'].dropna().unique()
missing = sorted([c for c in candidates if c not in present])
missing_df = pd.Series(missing, name='missing_component').to_frame()

# Display compact outputs: summary table and missing components (both as pandas objects)
summary_df, missing_df.head(200)

```
Out[2]:
```
```python
# Summarize previous aggregation results in plain English (compact)
missing_list = missing_df['missing_component'].tolist()
summary = (
    "Restricted metric.csv to candidate components (os_001..os_022, docker_001..docker_008, db_001..db_013).\n\n"
    "Top components by row_count (up to 20):\n"
    f"{summary_df.to_string(index=False)}\n\n"
    f"Missing candidate components not present in metric.csv (count {len(missing_list)}): "
    + ", ".join(missing_list) + "."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  row_count  unique_kpi_count            min_timestamp            max_timestamp
6   db_007       1053                46  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
12  db_013       1053                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
11  db_012       1051                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
10  db_011       1049                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
0   db_001       1048                45  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
7   db_008       1047                45  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
4   db_005       1047                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
8   db_009       1046                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
1   db_002       1046                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
3   db_004       1046                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
9   db_010       1045                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
5   db_006       1045                45  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
25  os_012        996                51  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
27  os_014        995                51  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
34  os_022        991                51  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
28  os_015        991                51  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
33  os_021        989                51  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
32  os_020        989                50  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
29  os_016        989                51  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
31  os_019        989                51  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST,   missing_component
0            os_001
1            os_002
2            os_003
3            os_004
4            os_005
5            os_009
6            os_010
7            os_017)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing metric variable and timezone if present; otherwise load/define
tz = pytz.timezone('Asia/Shanghai')

# Candidate sets
os_ids = [f"os_{i:03d}" for i in range(1, 23)]
docker_ids = [f"docker_{i:03d}" for i in range(1, 9)]
db_ids = [f"db_{i:03d}" for i in range(1, 14)]
candidates = os_ids + docker_ids + db_ids

# Determine which candidate components are present in metric.csv
present_in_metric = set(metric['cmdb_id'].dropna().unique())
candidate_present = sorted([c for c in candidates if c in present_in_metric])

# Filter metric to only candidate components that appear
metric_candidate = metric[metric['cmdb_id'].isin(candidate_present)].copy()

# --- 1) Compute global stats per (cmdb_id, kpi_name) using full series (global_p95, global_p5, total_count) ---
# Ensure value is numeric
metric_candidate['value'] = pd.to_numeric(metric_candidate['value'], errors='coerce')

global_stats = (
    metric_candidate
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        total_count='count',
        global_p95=lambda x: x.quantile(0.95),
        global_p5=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# --- 2) Filter to incident window in UTC+8: 2020-05-22 00:00:00 to 2020-05-22 00:30:00 inclusive ---
start_ts = pd.Timestamp('2020-05-22 00:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2020-05-22 00:30:00').tz_localize(tz)

# Convert epoch seconds to timezone-aware datetimes in UTC+8 for filtering
metric_candidate['dt'] = pd.to_datetime(metric_candidate['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

window = metric_candidate[(metric_candidate['dt'] >= start_ts) & (metric_candidate['dt'] <= end_ts)].copy()

# --- 3) Compute within-window aggregates per (cmdb_id, kpi_name) ---
window_stats = (
    window
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        window_count=('value', 'count'),
        window_min=('value', 'min'),
        window_max=('value', 'max'),
        window_first_ts=('dt', 'min'),
        window_last_ts=('dt', 'max')
    )
    .reset_index()
)

# Format timestamps to strings in UTC+8
window_stats['window_first_timestamp'] = window_stats['window_first_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
window_stats['window_last_timestamp'] = window_stats['window_last_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Drop the raw dt columns
window_stats = window_stats.drop(columns=['window_first_ts', 'window_last_ts'])

# --- 4) Merge global and window stats; keep only pairs that have window data (inner join) ---
merged = pd.merge(
    global_stats,
    window_stats,
    on=['cmdb_id', 'kpi_name'],
    how='inner'
)

# Select and order required columns
final_cols = [
    'cmdb_id', 'kpi_name',
    'global_p95', 'global_p5', 'total_count',
    'window_count', 'window_min', 'window_max',
    'window_first_timestamp', 'window_last_timestamp'
]
result_df = merged.loc[:, final_cols]

# Sort by window_count descending and limit to top 20 rows (compact)
result_df = result_df.sort_values('window_count', ascending=False).head(20)

# Display compact result
result_df

```
Out[3]:
```
# Summarize the aggregation results in plain English (compact)
top_n = result_df.head(3)
total_pairs_with_window = merged.shape[0]

lines = [
    f"Computed global p95/p5 and window aggregates for candidate (cmdb_id, kpi_name) pairs.",
    f"Total distinct pairs with data inside the incident window: {total_pairs_with_window}.",
    f"Incident window: 2020-05-22 00:00:00 to 2020-05-22 00:30:00 (UTC+8).",
    "",
    "Top 3 pairs by window_count:",
]

for _, r in top_n.iterrows():
    lines.append(
        f"- {r['cmdb_id']} | {r['kpi_name']} -> window_count={int(r['window_count'])}, "
        f"window_min={r['window_min']}, window_max={r['window_max']}, "
        f"global_p95={float(r['global_p95']):.6f}, global_p5={float(r['global_p5']):.6f}, "
        f"window_first={r['window_first_timestamp']}, window_last={r['window_last_timestamp']}"
    )

lines.append("")
lines.append("Notes: many top entries have full-window coverage (window_count=31); some DB metrics have 30 points ending at 00:29:00.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name  global_p95   global_p5  total_count  window_count  window_min  window_max   window_first_timestamp    window_last_timestamp
870   os_012              Disk_io_util    0.508376    0.000000           31            31    0.000000    1.000000  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
1035  os_015  Incoming_network_traffic    0.002886    0.000000           31            31    0.000000    0.003460  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
1349  os_022  Outgoing_network_traffic    0.062908    0.030158           31            31    0.002978    0.065872  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
721   os_007               Disk_rd_ios    0.000000    0.000000           31            31    0.000000    0.000000  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
322   db_008              CPU_free_pct   98.627099   97.367076           31            31   96.817903   98.656362  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
800   os_008                Send_total    0.000677    0.000516           31            31    0.000508    0.000683  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
1227  os_020              Disk_io_util    0.000000    0.000000           31            31    0.000000    0.000000  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
700   os_006                Send_total    0.000472    0.000306           31            31    0.000304    0.000474  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
896   os_012      Processor_load_5_min    0.014166    0.001667           31            31    0.001667    0.018333  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
5     db_001              CPU_free_pct   99.290966   98.182007           31            31   98.138415   99.357777  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
289   db_007             MEM_real_util   82.130000   82.100000           31            31   82.090000   82.130000  2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST
503   db_012              Call_Per_Sec  303.420000  164.300000           30            30  161.500000  439.600000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
504   db_012       DFParaWrite_Per_Sec    0.118500    0.040000           30            30    0.040000    0.170000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
505   db_012                    DbTime   15.410000    8.480000           30            30    8.100000   16.300000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
549   db_013       DFParaWrite_Per_Sec    0.065500    0.013500           30            30    0.000000    0.080000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
506   db_012              Exec_Per_Sec   59.790000   24.460000           30            30   23.400000   71.400000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
507   db_012                      Hang    1.000000    1.000000           30            30    1.000000    1.000000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
508   db_012       LFParaWrite_Per_Sec    0.205500    0.120000           30            30    0.100000    0.220000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
585   db_013               User_Commit    0.000000    0.000000           30            30    0.000000    0.190000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
548   db_013              Call_Per_Sec   74.840000   45.145000           30            30   45.000000   83.500000  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables where possible: metric, global_stats, tz, candidate_present, metric_candidate, window
# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# If global_stats or window not in memory (safety), recompute minimal pieces (but prefer reuse)
try:
    global_stats
    window
except NameError:
    # Recompute candidate_present and metric_candidate if needed
    os_ids = [f"os_{i:03d}" for i in range(1, 23)]
    docker_ids = [f"docker_{i:03d}" for i in range(1, 9)]
    db_ids = [f"db_{i:03d}" for i in range(1, 14)]
    candidates = os_ids + docker_ids + db_ids
    present_in_metric = set(metric['cmdb_id'].dropna().unique())
    candidate_present = sorted([c for c in candidates if c in present_in_metric])
    metric_candidate = metric[metric['cmdb_id'].isin(candidate_present)].copy()
    metric_candidate['value'] = pd.to_numeric(metric_candidate['value'], errors='coerce')
    global_stats = (
        metric_candidate
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(
            total_count='count',
            global_p95=lambda x: x.quantile(0.95),
            global_p5=lambda x: x.quantile(0.05)
        )
        .reset_index()
    )
    metric_candidate['dt'] = pd.to_datetime(metric_candidate['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_ts = pd.Timestamp('2020-05-22 00:00:00').tz_localize(tz)
    end_ts = pd.Timestamp('2020-05-22 00:30:00').tz_localize(tz)
    window = metric_candidate[(metric_candidate['dt'] >= start_ts) & (metric_candidate['dt'] <= end_ts)].copy()

# Join window rows with global thresholds (global_stats)
window_joined = window.merge(global_stats[['cmdb_id','kpi_name','global_p95','global_p5']], on=['cmdb_id','kpi_name'], how='left')

# Ensure numeric
window_joined['value'] = pd.to_numeric(window_joined['value'], errors='coerce')

# Mark anomalies
window_joined['is_high'] = (window_joined['value'] >= window_joined['global_p95']) & window_joined['global_p95'].notna()
window_joined['is_low'] = (window_joined['value'] <= window_joined['global_p5']) & window_joined['global_p5'].notna()
window_joined['is_anom'] = window_joined['is_high'] | window_joined['is_low']

# Compute excess/deficit
window_joined['excess_over_p95'] = np.where(window_joined['is_high'], window_joined['value'] - window_joined['global_p95'], np.nan)
window_joined['deficit_under_p5'] = np.where(window_joined['is_low'], window_joined['value'] - window_joined['global_p5'], np.nan)

# Aggregations per (cmdb_id, kpi_name)
grp = window_joined.groupby(['cmdb_id','kpi_name'])

agg_main = grp.agg(
    anomaly_count=('is_anom', 'sum'),
    high_count=('is_high', 'sum'),
    low_count=('is_low', 'sum'),
    window_count=('value', 'count'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
).reset_index()

# Earliest and latest anomaly timestamps (from anomalous rows)
anom_rows = window_joined[window_joined['is_anom']].copy()
if not anom_rows.empty:
    anom_times = anom_rows.groupby(['cmdb_id','kpi_name'])['dt'].agg(['min','max']).reset_index().rename(columns={'min':'earliest_anom_dt','max':'latest_anom_dt'})
else:
    anom_times = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_dt','latest_anom_dt'])

# Max excess over p95 and max deficit under p5
excess_agg = window_joined.groupby(['cmdb_id','kpi_name'])['excess_over_p95'].agg(
    max_excess_over_p95=lambda x: x.max(skipna=True)
).reset_index()
deficit_agg = window_joined.groupby(['cmdb_id','kpi_name'])['deficit_under_p5'].agg(
    max_deficit_under_p5=lambda x: x.min(skipna=True)  # min gives the largest negative deficit (most below p5)
).reset_index()

# Merge all pieces
merged_anoms = agg_main.merge(anom_times, on=['cmdb_id','kpi_name'], how='left') \
                       .merge(excess_agg, on=['cmdb_id','kpi_name'], how='left') \
                       .merge(deficit_agg, on=['cmdb_id','kpi_name'], how='left')

# Determine predominant_direction
def direction(row):
    if row['anomaly_count'] == 0:
        return 'none'
    if row['high_count'] > row['low_count']:
        return 'high'
    if row['low_count'] > row['high_count']:
        return 'low'
    return 'both'  # equal and >0

merged_anoms['predominant_direction'] = merged_anoms.apply(direction, axis=1)

# Format timestamps to strings (UTC+8)
merged_anoms['earliest_anomaly_timestamp'] = merged_anoms['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
merged_anoms['latest_anomaly_timestamp'] = merged_anoms['latest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and order columns per requirement
final_cols = [
    'cmdb_id','kpi_name',
    'anomaly_count','earliest_anomaly_timestamp','latest_anomaly_timestamp',
    'window_count','max_value_in_window','min_value_in_window',
    'predominant_direction','max_excess_over_p95','max_deficit_under_p5'
]
final_table = merged_anoms.loc[:, final_cols].sort_values('anomaly_count', ascending=False).head(20)

# Count distinct pairs with at least one anomaly
anomaly_pairs_count = int((merged_anoms['anomaly_count'] > 0).sum())

# Display compact outputs
final_table, anomaly_pairs_count

```
Out[4]:
```
```python
# Summarize the anomaly inspection results in plain English (compact)
top = final_table.head(5)
lines = [
    f"Total distinct (cmdb_id,kpi_name) pairs with ≥1 anomaly in the window: {anomaly_pairs_count}.",
    "Top 5 pairs by anomaly_count (cmdb_id | kpi_name -> anomalies, direction, window_count, earliest, latest):"
]
for _, r in top.iterrows():
    lines.append(
        f"- {r['cmdb_id']} | {r['kpi_name']} -> {int(r['anomaly_count'])} anomalies, "
        f"direction={r['predominant_direction']}, window_count={int(r['window_count'])}, "
        f"first={r['earliest_anomaly_timestamp']}, last={r['latest_anomaly_timestamp']}"
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                   kpi_name  anomaly_count earliest_anomaly_timestamp latest_anomaly_timestamp  window_count  max_value_in_window  min_value_in_window predominant_direction  max_excess_over_p95  max_deficit_under_p5
1227      os_020               Disk_io_util             31    2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST            31              0.00000              0.00000                  both                 0.00                   0.0
721       os_007                Disk_rd_ios             31    2020-05-22 00:00:00 CST  2020-05-22 00:30:00 CST            31              0.00000              0.00000                  both                 0.00                   0.0
247       db_006               On_Off_State             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              1.00000              1.00000                  both                 0.00                   0.0
241       db_006                  MEM_Total             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30            508.25241            508.25241                  both                 0.00                   0.0
1078      os_016                Disk_wr_ios             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              6.00000              5.00000                   low                 0.00                   0.0
588   docker_001              container_fgc             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
586       db_013        tnsping_result_time             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30             30.00000             10.00000                  high                10.00                   0.0
585       db_013                User_Commit             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.19000              0.00000                  high                 0.19                   0.0
888       os_012            Memory_used_pct             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30             91.21000             91.20000                   low                 0.00                   0.0
1092      os_016            Memory_used_pct             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30             26.48000             26.47000                   low                 0.00                   0.0
1074      os_016               Disk_io_util             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
299       db_007                   Row_Lock             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
595   docker_001  container_thread_used_pct             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
594   docker_001     container_thread_total             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
593   docker_001   container_thread_running             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
257       db_006                Sess_Active             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.30000              0.10000                   low                 0.10                   0.0
272       db_007                       AIOS             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.01000              0.00000                  high                 0.01                   0.0
244       db_006              MEM_real_util             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
615   docker_004              container_fgc             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              0.00000              0.00000                  both                 0.00                   0.0
281       db_007                       Hang             30    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST            30              1.00000              1.00000                  both                 0.00                   0.0, 1369)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure merged_anoms exists
try:
    merged_anoms
except NameError:
    raise RuntimeError("merged_anoms not found. Please run the anomaly aggregation steps first.")

df = merged_anoms.copy()

# Ensure numeric columns
df['anomaly_count'] = pd.to_numeric(df['anomaly_count'], errors='coerce').fillna(0).astype(int)
df['max_excess_over_p95'] = pd.to_numeric(df['max_excess_over_p95'], errors='coerce')
df['max_deficit_under_p5'] = pd.to_numeric(df['max_deficit_under_p5'], errors='coerce')

# Ensure earliest_anom_dt is datetime tz-aware (if present)
if 'earliest_anom_dt' in df.columns:
    df['earliest_anom_dt'] = pd.to_datetime(df['earliest_anom_dt']).dt.tz_convert(tz)
else:
    df['earliest_anom_dt'] = pd.NaT

# Helper to compute predominant direction per cmdb_id (tie -> join with '/')
def mode_directions(series):
    vc = series.value_counts()
    if vc.empty:
        return 'none'
    maxc = vc.max()
    modes = sorted(vc[vc == maxc].index.tolist())
    return '/'.join(modes)

grp = df.groupby('cmdb_id')

total_anomaly_count = grp['anomaly_count'].sum()
num_kpis_with_anomaly = grp.apply(lambda g: (g['anomaly_count'] > 0).sum())
max_excess = grp['max_excess_over_p95'].agg(lambda x: np.nan if x.dropna().empty else x.max(skipna=True))
min_deficit = grp['max_deficit_under_p5'].agg(lambda x: np.nan if x.dropna().empty else x.min(skipna=True))
earliest_anom = grp['earliest_anom_dt'].agg(lambda x: pd.NaT if x.dropna().empty else x.min())
pred_direction = grp['predominant_direction'].agg(mode_directions)

cmdb_summary_df = pd.concat([
    total_anomaly_count.rename('total_anomaly_count'),
    num_kpis_with_anomaly.rename('num_kpis_with_anomaly'),
    max_excess.rename('max_excess_over_p95'),
    min_deficit.rename('min_deficit_under_p5'),
    earliest_anom.rename('earliest_anom_dt'),
    pred_direction.rename('predominant_direction')
], axis=1).reset_index()

# Format earliest timestamp to string (UTC+8)
cmdb_summary_df['earliest_anomaly_timestamp'] = cmdb_summary_df['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
cmdb_summary_df['earliest_anomaly_timestamp'] = cmdb_summary_df['earliest_anomaly_timestamp'].fillna('')

# Select and order columns
cmdb_summary_df = cmdb_summary_df[
    ['cmdb_id', 'total_anomaly_count', 'num_kpis_with_anomaly',
     'max_excess_over_p95', 'min_deficit_under_p5',
     'earliest_anomaly_timestamp', 'predominant_direction']
]

# Sort by total_anomaly_count desc and limit to top 20
cmdb_summary_df = cmdb_summary_df.sort_values('total_anomaly_count', ascending=False).head(20)

# Separate list: components with max_excess_over_p95 >= 1.0
severe_df = cmdb_summary_df[cmdb_summary_df['max_excess_over_p95'].notna() & (cmdb_summary_df['max_excess_over_p95'] >= 1.0)]
severe_df = severe_df[['cmdb_id','max_excess_over_p95','earliest_anomaly_timestamp','total_anomaly_count']].sort_values(['max_excess_over_p95','total_anomaly_count'], ascending=[False, False]).head(20)

# Display compact outputs
cmdb_summary_df, severe_df

```
Out[5]:
```
```python
# Summarize per-cmdb anomaly aggregation results in plain English (compact)
# Reuse existing variables: cmdb_summary_df, severe_df, merged_anoms

# Compute total number of cmdb_ids that had at least one anomaly
cmdb_with_anom_count = int((merged_anoms.groupby('cmdb_id')['anomaly_count'].sum() > 0).sum())

# Top cmdb by total_anomaly_count from the computed summary (first row of cmdb_summary_df)
top_row = cmdb_summary_df.iloc[0]
top_cmdb = top_row['cmdb_id']
top_count = int(top_row['total_anomaly_count'])
top_kpis = int(top_row['num_kpis_with_anomaly'])
top_earliest = top_row['earliest_anomaly_timestamp']
top_direction = top_row['predominant_direction']

# Severe components with max_excess_over_p95 >= 1.0 (from severe_df)
severe_count = severe_df.shape[0]
severe_list = severe_df['cmdb_id'].astype(str).tolist()[:10]  # show up to first 10

summary = (
    f"Per-component anomaly summary:\n"
    f"- Number of components (cmdb_id) with ≥1 anomaly in the window: {cmdb_with_anom_count}.\n"
    f"- Top component by total anomalies: {top_cmdb} with {top_count} anomalies across {top_kpis} KPIs; "
    f"earliest anomaly at {top_earliest}; predominant_direction='{top_direction}'.\n"
    f"- Components with large excess (max_excess_over_p95 ≥ 1.0): {severe_count} found. "
    f"Examples (up to 10): {', '.join(severe_list)}.\n"
    f"- Note: many top components show 'both' as predominant_direction (high and low anomalies present), "
    f"and earliest anomalies start at 2020-05-22 00:00:00 (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  total_anomaly_count  num_kpis_with_anomaly  max_excess_over_p95  min_deficit_under_p5 earliest_anomaly_timestamp predominant_direction
5   db_006                  713                     45         1.166219e+04             -174.4300    2020-05-22 00:00:00 CST                  both
12  db_013                  684                     45         3.174463e+04               -8.1400    2020-05-22 00:00:00 CST                  both
0   db_001                  671                     45         1.234894e+04               -4.0200    2020-05-22 00:00:00 CST                  both
4   db_005                  658                     45         9.868535e+05              -56.2050    2020-05-22 00:00:00 CST                  both
10  db_011                  652                     45         1.061512e+06              -12.5100    2020-05-22 00:00:00 CST                  both
1   db_002                  640                     45         1.022114e+04              -56.7000    2020-05-22 00:00:00 CST                  both
22  os_007                  638                     50         7.600000e+00               -0.0381    2020-05-22 00:00:00 CST                  both
7   db_008                  632                     45         1.013070e+06              -64.9900    2020-05-22 00:00:00 CST                  both
8   db_009                  626                     45         5.893045e+03            -2090.7200    2020-05-22 00:00:00 CST                  both
23  os_008                  619                     50         2.450000e+00               -2.0000    2020-05-22 00:00:00 CST                  both
6   db_007                  611                     46         5.612315e+03              -26.5950    2020-05-22 00:00:00 CST                  both
3   db_004                  592                     45         3.211976e+04          -284724.0400    2020-05-22 00:00:00 CST                  both
21  os_006                  587                     51         3.800000e+00               -0.9000    2020-05-22 00:00:00 CST                  both
11  db_012                  585                     45         2.888081e+04              -73.3850    2020-05-22 00:00:00 CST                  both
9   db_010                  561                     45         3.826701e+04             -229.0300    2020-05-22 00:00:00 CST                  both
28  os_015                  541                     51         2.135000e+01              -14.9000    2020-05-22 00:00:00 CST                  both
26  os_013                  532                     51         8.500000e+00               -2.0000    2020-05-22 00:00:00 CST                  both
29  os_016                  528                     51         8.050000e+00              -10.0000    2020-05-22 00:00:00 CST                  both
24  os_011                  517                     50         1.950500e+00               -2.0000    2020-05-22 00:00:00 CST                  both
25  os_012                  498                     51         1.050000e+01               -5.9000    2020-05-22 00:00:00 CST                  both,    cmdb_id  max_excess_over_p95 earliest_anomaly_timestamp  total_anomaly_count
10  db_011         1.061512e+06    2020-05-22 00:00:00 CST                  652
7   db_008         1.013070e+06    2020-05-22 00:00:00 CST                  632
4   db_005         9.868535e+05    2020-05-22 00:00:00 CST                  658
9   db_010         3.826701e+04    2020-05-22 00:00:00 CST                  561
3   db_004         3.211976e+04    2020-05-22 00:00:00 CST                  592
12  db_013         3.174463e+04    2020-05-22 00:00:00 CST                  684
11  db_012         2.888081e+04    2020-05-22 00:00:00 CST                  585
0   db_001         1.234894e+04    2020-05-22 00:00:00 CST                  671
5   db_006         1.166219e+04    2020-05-22 00:00:00 CST                  713
1   db_002         1.022114e+04    2020-05-22 00:00:00 CST                  640
8   db_009         5.893045e+03    2020-05-22 00:00:00 CST                  626
6   db_007         5.612315e+03    2020-05-22 00:00:00 CST                  611
28  os_015         2.135000e+01    2020-05-22 00:00:00 CST                  541
25  os_012         1.050000e+01    2020-05-22 00:00:00 CST                  498
26  os_013         8.500000e+00    2020-05-22 00:00:00 CST                  532
29  os_016         8.050000e+00    2020-05-22 00:00:00 CST                  528
22  os_007         7.600000e+00    2020-05-22 00:00:00 CST                  638
21  os_006         3.800000e+00    2020-05-22 00:00:00 CST                  587
23  os_008         2.450000e+00    2020-05-22 00:00:00 CST                  619
24  os_011         1.950500e+00    2020-05-22 00:00:00 CST                  517)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'trace' DataFrame and timezone if present
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace is loaded
trace = trace.copy()

# Ensure numeric values
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) Compute global thresholds per (cmdb_id, trace_name) using full series (global_p95, global_p5, total_count)
global_trace = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        total_count='count',
        global_p95=lambda x: x.quantile(0.95),
        global_p5=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter to incident window (UTC+8) and mark anomalies using global thresholds (thresholds computed above)
trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start_ts = pd.Timestamp('2020-05-22 00:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2020-05-22 00:30:00').tz_localize(tz)

window_trace = trace[(trace['dt'] >= start_ts) & (trace['dt'] <= end_ts)].copy()

# Join thresholds (global) to window rows
window_joined = window_trace.merge(global_trace, on=['cmdb_id', 'trace_name'], how='left')

# Mark anomalies
window_joined['is_high'] = (window_joined['value'] >= window_joined['global_p95']) & window_joined['global_p95'].notna()
window_joined['is_low'] = (window_joined['value'] <= window_joined['global_p5']) & window_joined['global_p5'].notna()
window_joined['is_anom'] = window_joined['is_high'] | window_joined['is_low']

# 3) Aggregate anomalies by (cmdb_id, trace_name) to get counts and earliest/latest anomaly times per pair
pair_agg = (
    window_joined[window_joined['is_anom']]
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        anomaly_count=('is_anom', 'sum'),
        earliest_anom_dt=('dt', 'min'),
        latest_anom_dt=('dt', 'max')
    )
    .reset_index()
)

# If no anomalous rows, pair_agg will be empty; handle gracefully
if pair_agg.empty:
    # Empty results
    cmdb_agg_top20 = pd.DataFrame(columns=[
        'cmdb_id','total_anomaly_count','num_trace_names_with_anomaly',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','top_3_trace_names_by_anomaly_count'
    ])
    trace_agg_top20 = pd.DataFrame(columns=['trace_name','total_anomaly_count','earliest_anomaly_timestamp'])
else:
    # Aggregate per cmdb_id
    # total_anomaly_count and num_trace_names_with_anomaly
    cmdb_grp = pair_agg.groupby('cmdb_id')
    cmdb_total = cmdb_grp['anomaly_count'].sum().rename('total_anomaly_count')
    cmdb_num_names = cmdb_grp.apply(lambda g: g['trace_name'].nunique()).rename('num_trace_names_with_anomaly')
    cmdb_earliest = cmdb_grp['earliest_anom_dt'].min().rename('earliest_anom_dt')
    cmdb_latest = cmdb_grp['latest_anom_dt'].max().rename('latest_anom_dt')

    # top 3 trace_names by anomaly_count per cmdb_id -> compact list "name:count, name2:count"
    def top3_list(subdf):
        top = subdf.sort_values('anomaly_count', ascending=False).head(3)
        items = [f"{r['trace_name']}:{int(r['anomaly_count'])}" for _, r in top.iterrows()]
        return ", ".join(items)

    cmdb_top3 = cmdb_grp.apply(top3_list).rename('top_3_trace_names_by_anomaly_count')

    cmdb_agg = pd.concat([cmdb_total, cmdb_num_names, cmdb_earliest, cmdb_latest, cmdb_top3], axis=1).reset_index()

    # Format timestamps
    cmdb_agg['earliest_anomaly_timestamp'] = pd.to_datetime(cmdb_agg['earliest_anom_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    cmdb_agg['latest_anomaly_timestamp'] = pd.to_datetime(cmdb_agg['latest_anom_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    cmdb_agg_top20 = cmdb_agg.loc[:, [
        'cmdb_id','total_anomaly_count','num_trace_names_with_anomaly',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','top_3_trace_names_by_anomaly_count'
    ]].sort_values('total_anomaly_count', ascending=False).head(20)

    # Part B: top trace_name rows across all cmdb_id
    trace_grp = pair_agg.groupby('trace_name')
    trace_total = trace_grp['anomaly_count'].sum().rename('total_anomaly_count')
    trace_earliest = trace_grp['earliest_anom_dt'].min().rename('earliest_anom_dt')
    trace_agg = pd.concat([trace_total, trace_earliest], axis=1).reset_index()
    trace_agg['earliest_anomaly_timestamp'] = pd.to_datetime(trace_agg['earliest_anom_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    trace_agg_top20 = trace_agg.loc[:, ['trace_name','total_anomaly_count','earliest_anomaly_timestamp']].sort_values('total_anomaly_count', ascending=False).head(20)

# Display compact outputs (A) cmdb_agg_top20 and (B) trace_agg_top20
cmdb_agg_top20, trace_agg_top20

```
Out[6]:
```
```python
# Plain-English summary of trace anomaly detection results (compact)
summary = (
    "Trace anomaly detection (incident window: 2020-05-22 00:00:00 to 2020-05-22 00:30:00 UTC+8). "
    "Global thresholds (per cmdb_id,trace_name) were computed from full series (p95/p5). "
    "Points >=p95 or <=p5 were marked as anomalies.\n\n"
    "A) Top cmdb_id by total anomaly count (top examples, sorted):\n"
    "- docker_001: 313 anomalies (28 trace names with anomalies), earliest 2020-05-22 00:00:00, latest 00:29:00\n"
    "- docker_004: 307 anomalies (28 names), earliest 00:00:00, latest 00:29:00\n"
    "- docker_002: 306 anomalies (28 names)\n"
    "- docker_003: 305 anomalies (28 names)\n"
    "- docker_006: 196 anomalies (16 names)\n"
    "- docker_008: 194 anomalies (16 names)\n"
    "- docker_005: 190 anomalies (16 names)\n"
    "- docker_007: 189 anomalies (16 names)\n"
    "- os_022: 169 anomalies (16 names)\n"
    "- os_021: 168 anomalies (16 names)\n\n"
    "Most of these components show anomalies beginning at 2020-05-22 00:00:00 and many end at 00:29:00.\n\n"
    "B) Top trace_names across all cmdb_id by anomaly count (top examples):\n"
    "- trace.self.local.error_rate: 216 anomalies (earliest 2020-05-22 00:00:00)\n"
    "- trace.self.jdbc.error_rate: 204 anomalies\n"
    "- trace.self.jdbc.duration_p95: 188 anomalies\n"
    "- trace.self.csf.error_rate: 174 anomalies\n"
    "- trace.self.flyremote.error_rate: 120 anomalies\n"
    "- Several remoteprocess error_rate traces each with 60 anomalies, e.g. trace.from_os_021.remoteprocess.error_rate, trace.from_docker_004.remoteprocess.error_rate, trace.from_root.osb.error_rate, trace.from_docker_001.remoteprocess.error_rate, etc.\n"
    "- Additional notable trace metrics: trace.self.local.duration_p95 (48 anomalies), trace.self.local.row_count (34), trace.self.local.duration_mean (32).\n\n"
    "Summary takeaway: Docker hosts (docker_001..docker_004 etc.) show the highest number of trace anomalies during the incident window, and several 'trace.self.*' error_rate/duration metrics are the most frequently anomalous across the system."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_anomaly_count  num_trace_names_with_anomaly earliest_anomaly_timestamp latest_anomaly_timestamp                 top_3_trace_names_by_anomaly_count
0  docker_001                  313                            28    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_os_022.remoteprocess.error_rate:30,...
3  docker_004                  307                            28    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_os_021.remoteprocess.error_rate:30,...
1  docker_002                  306                            28    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_os_022.remoteprocess.error_rate:30,...
2  docker_003                  305                            28    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_os_021.remoteprocess.error_rate:30,...
5  docker_006                  196                            16    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_docker_001.remoteprocess.error_rate...
7  docker_008                  194                            16    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_docker_003.remoteprocess.error_rate...
4  docker_005                  190                            16    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_docker_001.remoteprocess.error_rate...
6  docker_007                  189                            16    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_docker_003.remoteprocess.error_rate...
9      os_022                  169                            16    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_root.osb.error_rate:30, trace.self....
8      os_021                  168                            16    2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST  trace.from_root.osb.error_rate:30, trace.self....,                                         trace_name  total_anomaly_count earliest_anomaly_timestamp
42                     trace.self.local.error_rate                  216    2020-05-22 00:00:00 CST
38                      trace.self.jdbc.error_rate                  204    2020-05-22 00:00:00 CST
37                    trace.self.jdbc.duration_p95                  188    2020-05-22 00:00:00 CST
30                       trace.self.csf.error_rate                  174    2020-05-22 00:00:00 CST
34                 trace.self.flyremote.error_rate                  120    2020-05-22 00:00:00 CST
18      trace.from_os_021.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
14  trace.from_docker_004.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
26                  trace.from_root.osb.error_rate                   60    2020-05-22 00:00:00 CST
2   trace.from_docker_001.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
6   trace.from_docker_002.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
10  trace.from_docker_003.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
22      trace.from_os_022.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
62    trace.to_docker_005.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
66    trace.to_docker_006.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
74    trace.to_docker_008.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
70    trace.to_docker_007.remoteprocess.error_rate                   60    2020-05-22 00:00:00 CST
41                   trace.self.local.duration_p95                   48    2020-05-22 00:00:00 CST
43                      trace.self.local.row_count                   34    2020-05-22 00:00:00 CST
40                  trace.self.local.duration_mean                   32    2020-05-22 00:00:00 CST
36                   trace.self.jdbc.duration_mean                   32    2020-05-22 00:00:00 CST)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing 'trace' if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric value and dt column (UTC->convert to UTC+8)
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
if 'dt' not in trace.columns:
    trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window bounds (inclusive) in UTC+8
start_ts = pd.Timestamp('2020-05-22 00:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2020-05-22 00:30:00').tz_localize(tz)

# Filter to window
trace_window = trace[(trace['dt'] >= start_ts) & (trace['dt'] <= end_ts)].copy()

# 1) trace_name values that contain substring 'db_' (case-sensitive)
mask_db = trace_window['trace_name'].str.contains('db_', na=False)
db_traces = trace_window[mask_db]

agg_db = (
    db_traces
    .groupby('trace_name')
    .agg(
        row_count=('trace_name', 'size'),
        distinct_cmdb_id_count=('cmdb_id', lambda x: x.dropna().nunique()),
        earliest_dt=('dt', 'min'),
        latest_dt=('dt', 'max')
    )
    .reset_index()
)

# Format timestamps to strings in UTC+8
agg_db['earliest_timestamp'] = agg_db['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg_db['latest_timestamp'] = agg_db['latest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

agg_db = agg_db.loc[:, ['trace_name', 'row_count', 'distinct_cmdb_id_count', 'earliest_timestamp', 'latest_timestamp']] \
               .sort_values('row_count', ascending=False).head(50)

# 2) trace_name values that start with 'trace.from_docker_' and contain 'remoteprocess'
mask_docker_remote = trace_window['trace_name'].str.startswith('trace.from_docker_', na=False) & trace_window['trace_name'].str.contains('remoteprocess', na=False)
docker_remote_traces = trace_window[mask_docker_remote]

agg_docker_remote = (
    docker_remote_traces
    .groupby('trace_name')
    .agg(
        row_count=('trace_name', 'size'),
        distinct_cmdb_id_count=('cmdb_id', lambda x: x.dropna().nunique()),
        earliest_dt=('dt', 'min'),
        latest_dt=('dt', 'max')
    )
    .reset_index()
)

agg_docker_remote['earliest_timestamp'] = agg_docker_remote['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg_docker_remote['latest_timestamp'] = agg_docker_remote['latest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

agg_docker_remote = agg_docker_remote.loc[:, ['trace_name', 'row_count', 'distinct_cmdb_id_count', 'earliest_timestamp', 'latest_timestamp']] \
                                     .sort_values('row_count', ascending=False).head(50)

# Return compact results
agg_db, agg_docker_remote

```
Out[7]:
```
- For trace_name values containing substring "db_": none were present in the incident window (2020-05-22 00:00:00 to 2020-05-22 00:30:00 UTC+8). The result set is empty.

- For trace_name values that start with "trace.from_docker_" and contain "remoteprocess": 16 distinct trace_name entries were present. Each of these has row_count = 60 (points within the window), distinct_cmdb_id_count = 2, earliest_timestamp = 2020-05-22 00:00:00 UTC+8, latest_timestamp = 2020-05-22 00:29:00 UTC+8. The trace_names (examples) are:
  - trace.from_docker_001.remoteprocess.duration_mean
  - trace.from_docker_001.remoteprocess.duration_p95
  - trace.from_docker_001.remoteprocess.error_rate
  - trace.from_docker_001.remoteprocess.row_count
  - trace.from_docker_002.remoteprocess.duration_mean
  - trace.from_docker_002.remoteprocess.duration_p95
  - trace.from_docker_002.remoteprocess.error_rate
  - trace.from_docker_002.remoteprocess.row_count
  - trace.from_docker_003.remoteprocess.duration_mean
  - trace.from_docker_003.remoteprocess.duration_p95
  - trace.from_docker_003.remoteprocess.error_rate
  - trace.from_docker_003.remoteprocess.row_count
  - trace.from_docker_004.remoteprocess.duration_mean
  - trace.from_docker_004.remoteprocess.duration_p95
  - trace.from_docker_004.remoteprocess.error_rate
  - trace.from_docker_004.remoteprocess.row_count

All listed remoteprocess traces show full-window coverage (60 points between 00:00 and 00:29 UTC+8) and are reported by two distinct cmdb_id values each.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [trace_name, row_count, distinct_cmdb_id_count, earliest_timestamp, latest_timestamp]
Index: [],                                            trace_name  row_count  distinct_cmdb_id_count       earliest_timestamp         latest_timestamp
0   trace.from_docker_001.remoteprocess.duration_mean         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
1    trace.from_docker_001.remoteprocess.duration_p95         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
2      trace.from_docker_001.remoteprocess.error_rate         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
3       trace.from_docker_001.remoteprocess.row_count         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
4   trace.from_docker_002.remoteprocess.duration_mean         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
5    trace.from_docker_002.remoteprocess.duration_p95         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
6      trace.from_docker_002.remoteprocess.error_rate         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
7       trace.from_docker_002.remoteprocess.row_count         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
8   trace.from_docker_003.remoteprocess.duration_mean         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
9    trace.from_docker_003.remoteprocess.duration_p95         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
10     trace.from_docker_003.remoteprocess.error_rate         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
11      trace.from_docker_003.remoteprocess.row_count         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
12  trace.from_docker_004.remoteprocess.duration_mean         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
13   trace.from_docker_004.remoteprocess.duration_p95         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
14     trace.from_docker_004.remoteprocess.error_rate         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST
15      trace.from_docker_004.remoteprocess.row_count         60                       2  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST)```
```

In [ ]:
# In[8]:

# Reuse merged_anoms and candidate lists from prior steps
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing merged_anoms (per (cmdb_id,kpi_name) anomaly aggregates)
df = merged_anoms.copy()

# Normalize column names we expect to exist
# merged_anoms contains: 'anomaly_count', 'earliest_anom_dt', 'max_excess_over_p95', 'max_deficit_under_p5',
# and window max/min columns named 'max_value_in_window' and 'min_value_in_window'
for col in ['anomaly_count','max_excess_over_p95','max_deficit_under_p5','earliest_anom_dt','min_value_in_window','max_value_in_window']:
    if col not in df.columns:
        df[col] = np.nan

# Restrict to candidate components present: prefixes db_, os_, docker_
mask_candidate = df['cmdb_id'].astype(str).str.startswith(('db_','os_','docker_'))
df_cand = df[mask_candidate].copy()

# Ensure numeric
df_cand['anomaly_count'] = pd.to_numeric(df_cand['anomaly_count'], errors='coerce').fillna(0).astype(int)
df_cand['max_excess_over_p95'] = pd.to_numeric(df_cand['max_excess_over_p95'], errors='coerce')
df_cand['max_deficit_under_p5'] = pd.to_numeric(df_cand['max_deficit_under_p5'], errors='coerce')

# Ensure earliest_anom_dt is datetime tz-aware (if present)
if 'earliest_anom_dt' in df_cand.columns:
    df_cand['earliest_anom_dt'] = pd.to_datetime(df_cand['earliest_anom_dt']).dt.tz_convert(tz)
else:
    df_cand['earliest_anom_dt'] = pd.NaT

# Per-component aggregations
grp = df_cand.groupby('cmdb_id')

total_anomaly_count = grp['anomaly_count'].sum().rename('total_anomaly_count')
num_kpis_with_anomaly = grp.apply(lambda g: (g['anomaly_count'] > 0).sum()).rename('num_kpis_with_anomaly')
max_excess = grp['max_excess_over_p95'].agg(lambda x: np.nan if x.dropna().empty else x.max(skipna=True)).rename('max_excess_over_p95')
min_deficit = grp['max_deficit_under_p5'].agg(lambda x: np.nan if x.dropna().empty else x.min(skipna=True)).rename('min_deficit_under_p5')
earliest_anom = grp['earliest_anom_dt'].agg(lambda x: pd.NaT if x.dropna().empty else x.min()).rename('earliest_anom_dt')

comp_summary = pd.concat([total_anomaly_count, num_kpis_with_anomaly, max_excess, min_deficit, earliest_anom], axis=1).reset_index()

# Add level column based on cmdb_id prefix
def infer_level(cid):
    if str(cid).startswith('db_'):
        return 'db'
    if str(cid).startswith('os_'):
        return 'os'
    if str(cid).startswith('docker_'):
        return 'docker'
    return 'other'

comp_summary['level'] = comp_summary['cmdb_id'].apply(infer_level)

# Format earliest timestamp
comp_summary['earliest_anomaly_timestamp'] = pd.to_datetime(comp_summary['earliest_anom_dt']).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
comp_summary['earliest_anomaly_timestamp'] = comp_summary['earliest_anomaly_timestamp'].fillna('')

# Select and order columns
comp_summary_out = comp_summary.loc[:, [
    'cmdb_id','level','total_anomaly_count','num_kpis_with_anomaly',
    'max_excess_over_p95','min_deficit_under_p5','earliest_anomaly_timestamp'
]]

# Sort by max_excess_over_p95 descending (NaNs go last) and limit top 20
comp_summary_out = comp_summary_out.sort_values('max_excess_over_p95', ascending=False, na_position='last').head(20)

# --- Part 2: For top 5 components by max_excess_over_p95, pick single KPI that produced that max_excess ---
top5_cmdb = comp_summary_out['cmdb_id'].head(5).tolist()

kpi_rows = []
for cid in top5_cmdb:
    sub = df_cand[df_cand['cmdb_id'] == cid].copy()
    # select KPI with largest max_excess_over_p95 (tie -> take first). If all NaN, skip.
    if sub['max_excess_over_p95'].dropna().empty:
        # fallback: pick row with largest anomaly_count
        chosen = sub.sort_values('anomaly_count', ascending=False).head(1)
    else:
        chosen = sub.sort_values('max_excess_over_p95', ascending=False).head(1)
    if not chosen.empty:
        r = chosen.iloc[0]
        kpi_rows.append({
            'cmdb_id': r['cmdb_id'],
            'kpi_name': r['kpi_name'],
            'anomaly_count': int(r['anomaly_count']),
            'earliest_anomaly_timestamp': (pd.to_datetime(r['earliest_anom_dt']).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(r['earliest_anom_dt']) else ''),
            'window_min': r.get('min_value_in_window', np.nan),
            'window_max': r.get('max_value_in_window', np.nan),
            'max_excess_over_p95': r.get('max_excess_over_p95', np.nan),
            'max_deficit_under_p5': r.get('max_deficit_under_p5', np.nan)
        })

kpi_top_df = pd.DataFrame(kpi_rows, columns=[
    'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp',
    'window_min','window_max','max_excess_over_p95','max_deficit_under_p5'
])

# Keep final outputs compact
comp_summary_out, kpi_top_df

```
Out[8]:
```
```python
# Summarize per-component severity and top-KPI contributors in plain English (compact)
summary = (
    "Per-component severity (top 20 sorted by max_excess_over_p95):\n"
    "- Dominated by DB nodes. Top five components by max_excess_over_p95 are:\n"
    "  1) db_011 — max_excess_over_p95 ≈ 1,061,511; total_anomaly_count=652; earliest anomaly 2020-05-22 00:00:00 (UTC+8).\n"
    "  2) db_008 — max_excess_over_p95 ≈ 1,013,070; total_anomaly_count=632; earliest anomaly 2020-05-22 00:00:00.\n"
    "  3) db_005 — max_excess_over_p95 ≈ 986,853; total_anomaly_count=658; earliest anomaly 2020-05-22 00:00:00.\n"
    "  4) db_003 — max_excess_over_p95 ≈ 319,901; total_anomaly_count=421; earliest anomaly 2020-05-22 00:00:00.\n"
    "  5) db_010 — max_excess_over_p95 ≈ 38,267;  total_anomaly_count=561; earliest anomaly 2020-05-22 00:00:00.\n"
    "  (Other DBs follow; several OS nodes appear later in the list with much smaller max_excess values; docker nodes show lower peak excesses.)\n\n"
    "KPI-level rows responsible for the top excess per component (one KPI per top-5 component):\n"
    "- db_011 | Redo_Per_Sec -> anomaly_count=4, earliest=2020-05-22 00:01:00, window_min=401.8, window_max=1,090,818.6, "
    "max_excess_over_p95≈1,061,511.845, max_deficit_under_p5=-12.510\n"
    "- db_008 | Redo_Per_Sec -> anomaly_count=4, earliest=2020-05-22 00:01:00, window_min=381.7, window_max=1,032,275.8, "
    "max_excess_over_p95≈1,013,070.270, max_deficit_under_p5=-64.990\n"
    "- db_005 | Redo_Per_Sec -> anomaly_count=4, earliest=2020-05-22 00:07:00, window_min=91.5, window_max=1,010,555.3, "
    "max_excess_over_p95≈986,853.465, max_deficit_under_p5≈-2.275\n"
    "- db_003 | Logic_Read_Per_Sec -> anomaly_count=4, earliest=2020-05-22 00:00:00, window_min=81.4, window_max=414,651.0, "
    "max_excess_over_p95≈319,901.425, max_deficit_under_p5≈-853.875\n"
    "- db_010 | Redo_Per_Sec -> anomaly_count=4, earliest=2020-05-22 00:01:00, window_min=137.8, window_max=40,117.6, "
    "max_excess_over_p95≈38,267.010, max_deficit_under_p5≈-22.415\n\n"
    "Key takeaways:\n"
    "- The most severe anomalies (largest excess over global p95) are on DB metrics, especially Redo_Per_Sec and Logic_Read_Per_Sec.\n"
    "- These KPI anomalies have relatively small anomaly counts (e.g., 4 points) but extremely large magnitudes when they occur.\n"
    "- Earliest anomalies for these top contributors occur at or shortly after 2020-05-22 00:00:00 (UTC+8).\n"
    "- OS and Docker components appear in the per-component summary but with much smaller max_excess values compared to the top DBs."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id   level  total_anomaly_count  num_kpis_with_anomaly  max_excess_over_p95  min_deficit_under_p5 earliest_anomaly_timestamp
10      db_011      db                  652                     45         1.061512e+06               -12.510    2020-05-22 00:00:00 CST
7       db_008      db                  632                     45         1.013070e+06               -64.990    2020-05-22 00:00:00 CST
4       db_005      db                  658                     45         9.868535e+05               -56.205    2020-05-22 00:00:00 CST
2       db_003      db                  421                     46         3.199014e+05              -853.875    2020-05-22 00:00:00 CST
9       db_010      db                  561                     45         3.826701e+04              -229.030    2020-05-22 00:00:00 CST
3       db_004      db                  592                     45         3.211976e+04           -284724.040    2020-05-22 00:00:00 CST
12      db_013      db                  684                     45         3.174463e+04                -8.140    2020-05-22 00:00:00 CST
11      db_012      db                  585                     45         2.888081e+04               -73.385    2020-05-22 00:00:00 CST
0       db_001      db                  671                     45         1.234894e+04                -4.020    2020-05-22 00:00:00 CST
5       db_006      db                  713                     45         1.166219e+04              -174.430    2020-05-22 00:00:00 CST
1       db_002      db                  640                     45         1.022114e+04               -56.700    2020-05-22 00:00:00 CST
8       db_009      db                  626                     45         5.893045e+03             -2090.720    2020-05-22 00:00:00 CST
6       db_007      db                  611                     46         5.612315e+03               -26.595    2020-05-22 00:00:00 CST
34      os_022      os                  465                     51         7.990000e+01                -9.900    2020-05-22 00:00:00 CST
33      os_021      os                  467                     51         7.435000e+01               -15.350    2020-05-22 00:00:00 CST
30      os_018      os                  405                     51         6.834253e+01               -56.100    2020-05-22 00:00:00 CST
32      os_020      os                  410                     50         2.490000e+01               -77.450    2020-05-22 00:00:00 CST
28      os_015      os                  541                     51         2.135000e+01               -14.900    2020-05-22 00:00:00 CST
31      os_019      os                  467                     51         2.102037e+01               -56.550    2020-05-22 00:00:00 CST
19  docker_007  docker                  261                      9         1.500000e+01                 0.000    2020-05-22 00:00:00 CST,   cmdb_id            kpi_name  anomaly_count earliest_anomaly_timestamp  window_min  window_max  max_excess_over_p95  max_deficit_under_p5
0  db_011        Redo_Per_Sec              4    2020-05-22 00:01:00 CST       401.8   1090818.6          1061511.845               -12.510
1  db_008        Redo_Per_Sec              4    2020-05-22 00:01:00 CST       381.7   1032275.8          1013070.270               -64.990
2  db_005        Redo_Per_Sec              4    2020-05-22 00:07:00 CST        91.5   1010555.3           986853.465                -2.275
3  db_003  Logic_Read_Per_Sec              4    2020-05-22 00:00:00 CST        81.4    414651.0           319901.425              -853.875
4  db_010        Redo_Per_Sec              4    2020-05-22 00:01:00 CST       137.8     40117.6            38267.010               -22.415)```
```

In [ ]:
# In[9]:

# Reuse existing variables where possible: merged_anoms, window, global_stats, tz
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required globals exist
try:
    merged_anoms
except NameError:
    raise RuntimeError("merged_anoms not found in the session. Run the anomaly aggregation steps first.")

# Use copies to avoid mutating originals
df = merged_anoms.copy()

# Ensure datetime column for earliest anomaly exists as tz-aware datetime
if 'earliest_anom_dt' in df.columns:
    df['earliest_anom_dt'] = pd.to_datetime(df['earliest_anom_dt']).dt.tz_convert(tz)
else:
    # try if string field exists
    if 'earliest_anomaly_timestamp' in df.columns:
        df['earliest_anom_dt'] = pd.to_datetime(df['earliest_anomaly_timestamp']).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
    else:
        df['earliest_anom_dt'] = pd.NaT

# Ensure we have window data (filtered incident window). If missing, try to reconstruct
try:
    window
except NameError:
    # Re-create window from metric if available
    if 'metric' in globals():
        metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        start_ts = pd.Timestamp('2020-05-22 00:00:00').tz_localize(tz)
        end_ts = pd.Timestamp('2020-05-22 00:30:00').tz_localize(tz)
        window = metric[(metric['dt'] >= start_ts) & (metric['dt'] <= end_ts)].copy()
    else:
        raise RuntimeError("Neither 'window' nor 'metric' available to compute window-level timestamps.")

# Compute per-(cmdb_id,kpi_name) window first/last timestamps and window min/max values (within incident window)
pair_window = (
    window
    .groupby(['cmdb_id','kpi_name'])
    .agg(
        window_first_dt=('dt','min'),
        window_last_dt=('dt','max'),
        window_min=('value','min'),
        window_max=('value','max')
    )
    .reset_index()
)

# Merge pair_window into df to ensure we have window_first/last and min/max
df2 = df.merge(pair_window, on=['cmdb_id','kpi_name'], how='left')

# Ensure global_p95/global_p5 available by merging global_stats if present
if 'global_stats' in globals():
    gs = global_stats[['cmdb_id','kpi_name','global_p95','global_p5']].copy()
    df2 = df2.merge(gs, on=['cmdb_id','kpi_name'], how='left')
else:
    # If not present, try to compute minimally from metric_candidate (should not recompute thresholds AFTER filtering)
    # But per rules, thresholds must be computed from full series earlier; here assume global_stats exists.
    df2['global_p95'] = np.nan
    df2['global_p5'] = np.nan

# Format timestamp strings for output
df2['earliest_anomaly_timestamp'] = df2['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
df2['earliest_anomaly_timestamp'] = df2['earliest_anomaly_timestamp'].fillna('')

df2['window_first_timestamp'] = pd.to_datetime(df2['window_first_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
df2['window_first_timestamp'] = df2['window_first_timestamp'].fillna('')

df2['window_last_timestamp'] = pd.to_datetime(df2['window_last_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
df2['window_last_timestamp'] = df2['window_last_timestamp'].fillna('')

# Select and prepare first output: top 10 KPI rows for cmdb_id == 'db_006'
db6 = df2[df2['cmdb_id'] == 'db_006'].copy()

# Ensure numeric conversions
for col in ['max_excess_over_p95','max_deficit_under_p5','global_p95','global_p5','window_min','window_max']:
    if col in db6.columns:
        db6[col] = pd.to_numeric(db6[col], errors='coerce')
    else:
        db6[col] = np.nan

# Prepare result columns
cols_out = [
    'kpi_name','anomaly_count','earliest_anomaly_timestamp',
    'window_first_timestamp','window_last_timestamp',
    'window_min','window_max','global_p95','global_p5',
    'max_excess_over_p95','max_deficit_under_p5'
]

db6_out = db6.loc[:, cols_out].sort_values('max_excess_over_p95', ascending=False).head(10)

# Second output: single KPI row across all components with overall earliest anomaly_timestamp among rows with max_excess_over_p95 >= 1000
df2['max_excess_over_p95'] = pd.to_numeric(df2.get('max_excess_over_p95', np.nan), errors='coerce')
candidates = df2[df2['max_excess_over_p95'] >= 1000].copy()

# Ensure earliest_anom_dt exists for sorting
candidates['earliest_anom_dt'] = pd.to_datetime(candidates['earliest_anom_dt'] if 'earliest_anom_dt' in candidates.columns else candidates['earliest_anom_dt'])

if not candidates.empty:
    earliest_row = candidates.sort_values('earliest_anom_dt', ascending=True).head(1)
    # format output
    earliest_out = earliest_row.loc[:, ['cmdb_id','kpi_name','earliest_anom_dt','max_excess_over_p95','anomaly_count']].copy()
    earliest_out['earliest_anomaly_timestamp'] = pd.to_datetime(earliest_out['earliest_anom_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    earliest_out = earliest_out.loc[:, ['cmdb_id','kpi_name','earliest_anomaly_timestamp','max_excess_over_p95','anomaly_count']]
else:
    earliest_out = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp','max_excess_over_p95','anomaly_count'])

# Display compact outputs
db6_out, earliest_out

```
Out[9]:
```
```python
# Summarize the requested results in plain English (compact)
try:
    db6 = db6_out.copy()
    earliest = earliest_out.copy()
except NameError:
    raise RuntimeError("Expected results (db6_out, earliest_out) not found in the session.")

lines = []
lines.append("Top 10 KPIs for db_006 (sorted by max_excess_over_p95):")
for _, r in db6.iterrows():
    k = r['kpi_name']
    ac = int(r['anomaly_count']) if pd.notna(r['anomaly_count']) else 0
    ea = r['earliest_anomaly_timestamp'] or ''
    wf = r['window_first_timestamp'] or ''
    wl = r['window_last_timestamp'] or ''
    wmin = r['window_min']
    wmax = r['window_max']
    gp95 = r['global_p95']
    gp5 = r['global_p5']
    mex = r['max_excess_over_p95']
    mdf = r['max_deficit_under_p5']
    # format numbers compactly
    def fmt(x):
        if pd.isna(x):
            return "NA"
        if isinstance(x, (int, np.integer)):
            return f"{x}"
        try:
            xv = float(x)
            if abs(xv) >= 1000:
                return f"{xv:,.0f}"
            return f"{xv:.3f}"
        except:
            return str(x)
    lines.append(f"- {k}: anomaly_count={ac}, earliest={ea}, window_first={wf}, window_last={wl}, window_min={fmt(wmin)}, window_max={fmt(wmax)}, global_p95={fmt(gp95)}, global_p5={fmt(gp5)}, max_excess={fmt(mex)}, max_deficit={fmt(mdf)}")

# Second short output: earliest KPI with max_excess_over_p95 >= 1000
lines.append("")
if earliest.shape[0] == 0:
    lines.append("No KPI found with max_excess_over_p95 >= 1000.")
else:
    r = earliest.iloc[0]
    cid = r['cmdb_id']
    kpi = r['kpi_name']
    ea = r['earliest_anomaly_timestamp']
    mex = r['max_excess_over_p95']
    ac = int(r['anomaly_count'])
    # format mex
    def fmt2(x):
        try:
            xv = float(x)
            if abs(xv) >= 1000:
                return f"{xv:,.0f}"
            return f"{xv:.3f}"
        except:
            return str(x)
    lines.append("Earliest KPI (among rows with max_excess_over_p95 >= 1000):")
    lines.append(f"- {cid} | {kpi} -> earliest={ea}, max_excess_over_p95={fmt2(mex)}, anomaly_count={ac}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  kpi_name  anomaly_count earliest_anomaly_timestamp   window_first_timestamp    window_last_timestamp  window_min  window_max  global_p95  global_p5  max_excess_over_p95  max_deficit_under_p5
253           Redo_Per_Sec             21    2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST         0.0    24514.70  12852.5100      0.000           11662.1900                 0.000
239     Logic_Read_Per_Sec              4    2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST       753.8     2441.30   2290.2650    928.230             151.0350              -174.430
235           Exec_Per_Sec              4    2020-05-22 00:01:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST       132.2      279.60    258.7800    170.275              20.8200               -38.075
234                 DbTime              5    2020-05-22 00:01:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST        10.6       32.30     23.8750     12.400               8.4250                -1.800
232           Call_Per_Sec              4    2020-05-22 00:01:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST       372.2      571.70    564.9700    486.735               6.7300              -114.535
250  Physical_Read_Per_Sec             29    2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST         0.0       75.40     70.6550      0.000               4.7450                 0.000
238         LFSync_Per_Sec             28    2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST         0.0        3.17      1.8785      0.000               1.2915                 0.000
262            TPS_Per_Sec              4    2020-05-22 00:01:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST        60.5       93.70     92.6200     79.510               1.0800               -19.010
237    LFParaWrite_Per_Sec             21    2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST         0.0        1.89      1.6980      0.000               0.1920                 0.000
257            Sess_Active             30    2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST  2020-05-22 00:29:00 CST         0.1        0.30      0.2000      0.100               0.1000                 0.000,     cmdb_id      kpi_name earliest_anomaly_timestamp  max_excess_over_p95  anomaly_count
569  db_013  Redo_Per_Sec    2020-05-22 00:00:00 CST             31744.63              4)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing data
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric DataFrame exists
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure datetime column in metric (UTC->UTC+8)
if 'dt' not in metric.columns:
    metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window bounds (inclusive)
start_ts = pd.Timestamp('2020-05-22 00:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2020-05-22 00:30:00').tz_localize(tz)

# KPIs to inspect for db_006
target_cmdb = 'db_006'
kpis = ['Redo_Per_Sec', 'Logic_Read_Per_Sec']

# Ensure global thresholds exist (global_stats). If missing, compute from full metric series per rule 9.
try:
    global_stats
except NameError:
    global_stats = (
        metric
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(total_count='count',
             global_p95=lambda x: x.quantile(0.95),
             global_p5=lambda x: x.quantile(0.05))
        .reset_index()
    )

# Prepare container for segments
segments_list = []

for k in kpis:
    # Get global_p95 for this pair
    row = global_stats[(global_stats['cmdb_id']==target_cmdb) & (global_stats['kpi_name']==k)]
    if row.empty:
        gp95 = np.nan
    else:
        gp95 = float(row['global_p95'].iloc[0])

    # Filter metric to the incident window and this KPI/cmdb
    dfw = metric[(metric['cmdb_id']==target_cmdb) & (metric['kpi_name']==k)].copy()
    if dfw.empty:
        # no data
        segs_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
            'segment_length_in_points','max_value_in_segment','max_excess_over_p95_in_segment'
        ])
        segments_list.append(segs_df)
        continue

    # Keep only window rows
    dfw = dfw[(dfw['dt'] >= start_ts) & (dfw['dt'] <= end_ts)].sort_values('dt').reset_index(drop=True)
    if dfw.empty:
        segs_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
            'segment_length_in_points','max_value_in_segment','max_excess_over_p95_in_segment'
        ])
        segments_list.append(segs_df)
        continue

    # Ensure numeric
    dfw['value'] = pd.to_numeric(dfw['value'], errors='coerce')

    # Mark high anomalies: value >= global_p95 (per instruction)
    dfw['is_anom'] = False
    if not np.isnan(gp95):
        dfw['is_anom'] = dfw['value'] >= gp95

    # Compute time diff to detect gaps larger than 60s
    dfw['dt_diff_s'] = dfw['dt'].diff().dt.total_seconds().fillna(0)

    # New segment starts when is_anom True and (previous was not anom OR time gap > 60s)
    dfw['prev_anom'] = dfw['is_anom'].shift(fill_value=False)
    dfw['new_seg'] = (dfw['is_anom']) & (~dfw['prev_anom'] | (dfw['dt_diff_s'] > 60))

    # Segment id as cumulative sum of new_seg (only meaningful where is_anom True)
    dfw['seg_id'] = dfw['new_seg'].cumsum()
    # Set seg_id to NaN for non-anom rows to ignore them
    dfw.loc[~dfw['is_anom'], 'seg_id'] = np.nan

    # Group by seg_id (drop NaN)
    seg_groups = dfw.dropna(subset=['seg_id']).groupby('seg_id')

    seg_rows = []
    for seg_id, g in seg_groups:
        start_ts_seg = g['dt'].min()
        end_ts_seg = g['dt'].max()
        length = int(g.shape[0])
        max_val = float(g['value'].max())
        # compute max excess over global_p95 in this segment
        if np.isnan(gp95):
            max_excess = np.nan
        else:
            max_excess = float((g['value'] - gp95).max())
        seg_rows.append({
            'cmdb_id': target_cmdb,
            'kpi_name': k,
            'segment_start_timestamp': start_ts_seg.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'),
            'segment_end_timestamp': end_ts_seg.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'),
            'segment_length_in_points': length,
            'max_value_in_segment': max_val,
            'max_excess_over_p95_in_segment': max_excess
        })

    segs_df = pd.DataFrame(seg_rows, columns=[
        'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
        'segment_length_in_points','max_value_in_segment','max_excess_over_p95_in_segment'
    ])

    # Limit to at most 10 segments (they are in chronological order because seg_id increases)
    if not segs_df.empty:
        segs_df = segs_df.head(10)

    segments_list.append(segs_df)

# Combine segments for both KPIs into one compact DataFrame
segments_combined = pd.concat(segments_list, ignore_index=True) if segments_list else pd.DataFrame(columns=[
    'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
    'segment_length_in_points','max_value_in_segment','max_excess_over_p95_in_segment'
])

# --- Second short output: earliest anomalous (cmdb_id,kpi_name) among rows with max_excess_over_p95 >= 1000 ---
# Use merged_anoms (per-(cmdb,kpi) anomaly table) which should have earliest_anom_dt and max_excess_over_p95
try:
    ma = merged_anoms.copy()
except NameError:
    # fallback: try to reconstruct minimal merged_anoms (not expected)
    ma = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_dt','max_excess_over_p95','anomaly_count'])

# Ensure datetime tz-aware
if 'earliest_anom_dt' in ma.columns:
    ma['earliest_anom_dt'] = pd.to_datetime(ma['earliest_anom_dt']).dt.tz_convert(tz)
else:
    ma['earliest_anom_dt'] = pd.NaT

ma['max_excess_over_p95'] = pd.to_numeric(ma.get('max_excess_over_p95', np.nan), errors='coerce')

cands = ma[ma['max_excess_over_p95'] >= 1000].copy()
if not cands.empty:
    earliest_kpi_row = cands.sort_values('earliest_anom_dt', ascending=True).head(1)
    earliest_kpi_out = earliest_kpi_row.loc[:, ['cmdb_id','kpi_name','earliest_anom_dt','max_excess_over_p95','anomaly_count']].copy()
    earliest_kpi_out['earliest_anomaly_timestamp'] = pd.to_datetime(earliest_kpi_out['earliest_anom_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    earliest_kpi_out = earliest_kpi_out.loc[:, ['cmdb_id','kpi_name','earliest_anomaly_timestamp','max_excess_over_p95','anomaly_count']]
else:
    earliest_kpi_out = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp','max_excess_over_p95','anomaly_count'])

# Final compact outputs: segments_combined (limited rows) and earliest_kpi_out (single row)
segments_combined, earliest_kpi_out

```
Out[10]:
```
```python
# Plain-English summary of fault identification results for db_006 and the earliest severe KPI
summary = (
    "db_006 anomaly segments (incident window 2020-05-22 00:00:00 to 00:30:00 UTC+8):\n"
    "- Redo_Per_Sec: two isolated anomalous points:\n"
    "  * 2020-05-22 00:00:00 — length 1, max_value=24,514.7, max_excess_over_p95≈11,662.19\n"
    "  * 2020-05-22 00:16:00 — length 1, max_value=22,844.4, max_excess_over_p95≈9,991.89\n"
    "- Logic_Read_Per_Sec: two isolated anomalous points:\n"
    "  * 2020-05-22 00:00:00 — length 1, max_value=2,441.3, max_excess_over_p95≈151.04\n"
    "  * 2020-05-22 00:16:00 — length 1, max_value=2,383.1, max_excess_over_p95≈92.84\n\n"
    "Earliest severe KPI across all DBs (max_excess_over_p95 ≥ 1000):\n"
    "- db_013 | Redo_Per_Sec — earliest anomalous timestamp 2020-05-22 00:00:00 UTC+8, "
    "max_excess_over_p95≈31,744.63, anomaly_count=4.\n\n"
    "Interpretation: db_006 shows brief, high-magnitude single-point spikes in Redo_Per_Sec and smaller spikes in Logic_Read_Per_Sec at 00:00 and 00:16. "
    "However, even earlier and larger excesses were observed on db_013 (Redo_Per_Sec) at 00:00."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id            kpi_name  segment_start_timestamp    segment_end_timestamp  segment_length_in_points  max_value_in_segment  max_excess_over_p95_in_segment
0  db_006        Redo_Per_Sec  2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST                         1               24514.7                       11662.190
1  db_006        Redo_Per_Sec  2020-05-22 00:16:00 CST  2020-05-22 00:16:00 CST                         1               22844.4                        9991.890
2  db_006  Logic_Read_Per_Sec  2020-05-22 00:00:00 CST  2020-05-22 00:00:00 CST                         1                2441.3                         151.035
3  db_006  Logic_Read_Per_Sec  2020-05-22 00:16:00 CST  2020-05-22 00:16:00 CST                         1                2383.1                          92.835,     cmdb_id      kpi_name earliest_anomaly_timestamp  max_excess_over_p95  anomaly_count
569  db_013  Redo_Per_Sec    2020-05-22 00:00:00 CST             31744.63              4)```
```